In [12]:
import requests
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text
import os

In [3]:
load_dotenv()

api_key = os.getenv("OPENWEATHER_API_KEY")

In [28]:
url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Incheon",
    "appid": api_key,
    "units": "metric"
}

response = requests.get(url, params=params)

In [29]:
weather = response.json()
weather



{'coord': {'lon': 126.4161, 'lat': 37.45},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 20.6,
  'feels_like': 21.22,
  'temp_min': 20.6,
  'temp_max': 20.6,
  'pressure': 1011,
  'humidity': 96,
  'sea_level': 1011,
  'grnd_level': 1011},
 'visibility': 10000,
 'wind': {'speed': 4.09, 'deg': 183, 'gust': 4.9},
 'clouds': {'all': 100},
 'dt': 1783044546,
 'sys': {'country': 'KR', 'sunrise': 1783023457, 'sunset': 1783076340},
 'timezone': 32400,
 'id': 1843561,
 'name': 'Incheon',
 'cod': 200}

In [47]:
load_dotenv()
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

In [48]:
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)

In [49]:
create_cities_table = """
CREATE TABLE IF NOT EXISTS cities(
    city_id SERIAL PRIMARY KEY,
    openweather_id INTEGER UNIQUE,
    city_name VARCHAR(50) NOT NULL,
    country CHAR(2) NOT NULL,
    latitude DECIMAL(8,5),
    longitude DECIMAL(8,5)
);
"""



In [50]:
#creating the table if not exists
with engine.begin() as conn:
    result = conn.execute(text(create_cities_table))


In [ ]:
# from sqlalchemy import text

# drop_table_sql = """
# DROP TABLE IF EXISTS observations;
# """

# with engine.begin() as conn:
#     conn.execute(text(drop_table_sql))

In [ ]:
insert_city_sql = """
INSERT INTO cities (
    openweather_id,
    city_name,
    country,
    latitude,
    longitude
)
VALUES (
    :openweather_id,
    :city_name,
    :country,
    :latitude,
    :longitude
)

RETURNING city_id
"""

In [52]:
city_param = {
    "openweather_id": weather["id"],
    "city_name": weather["name"],
    "country": weather["sys"]["country"],
    "latitude": weather["coord"]["lat"],
    "longitude": weather["coord"]["lon"]
}
city_param


{'openweather_id': 1843561,
 'city_name': 'Incheon',
 'country': 'KR',
 'latitude': 37.45,
 'longitude': 126.4161}

In [ ]:
get_city_id = """
SELECT city_id 
FROM cities
WHERE openweather_id = :openweather_id
"""

with engine.begin() as conn:
    result = conn.execute(text(get_city_id),city_param)
    city_id = result.scalar()

    if city_id == None:
        result = conn.execute(text(insert_city_sql),city_param)
        city_id = result.scalar()
        


2

In [61]:
see_city = """
SELECT * FROM cities
"""

with engine.connect() as conn:
    results = conn.execute(text(see_city))
    r = results.fetchall()
    print(r)


[(1, 1835848, 'Seoul', 'KR', Decimal('37.56830'), Decimal('126.97780')), (2, 1843561, 'Incheon', 'KR', Decimal('37.45000'), Decimal('126.41610'))]
